In [84]:
!pip install pyspark

In [85]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, count

In [86]:
spark = SparkSession.builder \
    .appName("MovieRatingsAnalytics") \
    .getOrCreate()

In [87]:
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", LongType(), True)
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

In [88]:
ratings_df = spark.read.csv(
    "rating.csv",
    header=True,
    schema=ratings_schema,
    nullValue="",
    emptyValue=""
)

movies_df = spark.read.csv(
    "movie.csv",
    header=True,
    schema=movies_schema,
    nullValue="",
    emptyValue=""
)


print("Ratings Dataset")
ratings_df.show(5)

print("Movies Dataset")
movies_df.show(5)


print("Ratings Schema")
ratings_df.printSchema()

print("Movies Schema")
movies_df.printSchema()

Ratings Dataset
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      2|   3.5|     NULL|
|     1|     29|   3.5|     NULL|
|     1|     32|   3.5|     NULL|
|     1|     47|   3.5|     NULL|
|     1|     50|   3.5|     NULL|
+------+-------+------+---------+
only showing top 5 rows
Movies Dataset
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
Ratings Schema
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timest

In [89]:
print("Null Value Counts")

Null Value Counts


In [90]:
ratings_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ratings_df.columns
]).show()

movies_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in movies_df.columns
]).show()

ratings_duplicates = ratings_df.count() - ratings_df.dropDuplicates(
    ["userId", "movieId"]
).count()

movies_duplicates = movies_df.count() - movies_df.dropDuplicates(
    ["movieId"]
).count()

print("Duplicate Ratings Count:", ratings_duplicates)
print("Duplicate Movies Count:", movies_duplicates)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     0|      0|     0| 20000263|
+------+-------+------+---------+

+-------+-----+------+
|movieId|title|genres|
+-------+-----+------+
|      0|    0|     0|
+-------+-----+------+

Duplicate Ratings Count: 0
Duplicate Movies Count: 0


In [91]:
print("Invalid Ratings (Outside 1-5 Range):")

ratings_df.filter(
    (col("rating") < 1.0) |
    (col("rating") > 5.0)
).show()

spark.stop()

Invalid Ratings (Outside 1-5 Range):
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|    11|    286|   0.5|     NULL|
|    11|    671|   0.5|     NULL|
|    11|   1077|   0.5|     NULL|
|    11|   1977|   0.5|     NULL|
|    11|   2107|   0.5|     NULL|
|    11|   2683|   0.5|     NULL|
|    11|   2990|   0.5|     NULL|
|    11|   3396|   0.5|     NULL|
|    11|  27793|   0.5|     NULL|
|    11|  34334|   0.5|     NULL|
|    11|  44225|   0.5|     NULL|
|    11|  44828|   0.5|     NULL|
|    11|  59376|   0.5|     NULL|
|    11|  64508|   0.5|     NULL|
|    11|  64999|   0.5|     NULL|
|    11|  67799|   0.5|     NULL|
|    11|  68263|   0.5|     NULL|
|    11|  70305|   0.5|     NULL|
|    30|    168|   0.5|     NULL|
|    31|    527|   0.5|     NULL|
+------+-------+------+---------+
only showing top 20 rows
